# 40× experiment

Patient-level, leakage-free evaluation of five odontogenic pathologies using ImageNet-pretrained DenseNet-121 and AlexNet.

Key safeguards:

- patient groups are created before splitting;
- the split is performed at patient/specimen level;
- PCA outlier detection is fitted on training images only;
- augmentation is applied only to the training set;
- validation and test images are never augmented or removed;
- the final metrics are computed on the held-out test patients;
- three independent repetitions are used for the principal comparison;
- a focused DenseNet-121 ablation is generated with and without PCA-based outlier removal.


In [ ]:
# Colab setup
!pip -q install torch torchvision scikit-learn seaborn pandas pillow

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from pathlib import Path
import os, re, json, math, random, copy, warnings
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix,
    precision_recall_fscore_support, log_loss, roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

warnings.filterwarnings('ignore')

DATA_ROOT = Path('/content/drive/MyDrive/GorlinIA/A_D_K_CROPPED')
OUTPUT_ROOT = Path('/content/drive/MyDrive/GorlinIA/Corrected_40X_results')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

CLASS_DIRS = {
    'Ameloblastoma': 'AMELOBLASTOMA',
    'Dentigerous cyst': 'DENTIGEROUS',
    'Odontogenic keratocyst': 'KERATOCYST',
    'Orthokeratinized odontogenic cyst': 'ORTHOKERATINIZED',
    'Hyperplastic dental follicle': 'DENTAL',
}
CLASS_NAMES = list(CLASS_DIRS)
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

N_REPEATS = 3
MAX_EPOCHS = 30
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 5
PCA_THRESHOLD_SD = 1.5
IMAGE_SIZE = 224
NUM_WORKERS = 2
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


## 1. Build the patient-level manifest

For the three legacy folders, each consecutive block of 20 images is treated as one patient. For classes already organized into case folders, the folder name is used as the patient identifier. A final incomplete block is retained as one additional patient.


In [ ]:
VALID_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp', '.heic', '.heif'}

def natural_key(path):
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r'(\d+)', path.name)]

def image_files(folder):
    if not folder.exists():
        return []
    return sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS], key=natural_key)

def discover_class_images(class_name, folder_name):
    class_root = DATA_ROOT / folder_name
    records = []

    # OOC and HDF are already arranged in explicit case folders. The three
    # legacy classes use consecutive blocks of 20 files in their direct 40X folder.
    use_explicit_case_folders = class_name in {
        'Orthokeratinized odontogenic cyst', 'Hyperplastic dental follicle'
    }
    case_folders = ([p for p in class_root.iterdir() if p.is_dir() and (p / '40X').exists()]
                    if use_explicit_case_folders else [])
    if case_folders:
        for case_dir in sorted(case_folders, key=lambda p: p.name):
            files = image_files(case_dir / '40X')
            for order, path in enumerate(files):
                records.append({
                    'path': str(path), 'class_name': class_name,
                    'label': CLASS_TO_IDX[class_name],
                    'patient_id': f'{folder_name}_{case_dir.name}',
                    'image_order': order,
                })
        return records

    # Legacy folders: every 20 numerically ordered images correspond to one patient.
    files = image_files(class_root / '40X')
    for order, path in enumerate(files):
        patient_number = order // 20 + 1
        records.append({
            'path': str(path), 'class_name': class_name,
            'label': CLASS_TO_IDX[class_name],
            'patient_id': f'{folder_name}_P{patient_number:03d}',
            'image_order': order,
        })
    return records

records = []
for class_name, folder_name in CLASS_DIRS.items():
    records.extend(discover_class_images(class_name, folder_name))

manifest = pd.DataFrame(records)
assert len(manifest), f'No images found under {DATA_ROOT}'
assert manifest.groupby('patient_id')['label'].nunique().max() == 1

display(manifest.groupby('class_name').agg(images=('path', 'size'), patients=('patient_id', 'nunique')))
manifest.to_csv(OUTPUT_ROOT / 'manifest_40X.csv', index=False)


## 2. Patient-level 60/20/20 split

Splitting is performed independently within each diagnostic class so that every class contributes patients to training, validation, and testing. No patient can occur in more than one subset.


In [ ]:
def split_patients(df, seed, train_fraction=0.60, val_fraction=0.20):
    rng = np.random.default_rng(seed)
    assignments = []
    for class_name, class_df in df.groupby('class_name'):
        patients = class_df['patient_id'].drop_duplicates().to_numpy()
        rng.shuffle(patients)
        n = len(patients)
        if n < 5:
            raise ValueError(f'{class_name} has only {n} patients; at least 3 are required.')
        n_train = max(1, int(round(n * train_fraction)))
        n_val = max(1, int(round(n * val_fraction)))
        if n_train + n_val >= n:
            n_train = n - 2
            n_val = 1
        split_map = {p: 'train' for p in patients[:n_train]}
        split_map.update({p: 'validation' for p in patients[n_train:n_train+n_val]})
        split_map.update({p: 'test' for p in patients[n_train+n_val:]})
        assignments.extend([(p, split_map[p]) for p in patients])

    assignment_df = pd.DataFrame(assignments, columns=['patient_id', 'split'])
    result = df.merge(assignment_df, on='patient_id', how='left')
    overlap = result.groupby('patient_id')['split'].nunique().max()
    assert overlap == 1, 'Patient leakage detected.'
    return result

example_split = split_patients(manifest, seed=42)
display(example_split.groupby(['class_name', 'split'])['patient_id'].nunique().unstack(fill_value=0))


## 3. Training-only PCA outlier removal

PCA is fitted separately within each diagnostic class using only training images. Images are converted to grayscale, resized to 200×200, normalized to [0,1], and projected onto the first two principal components. A training image is flagged when either score lies outside mean ± 1.5 standard deviations. Validation and test observations are never removed.


In [ ]:
def pca_vector(path):
    with Image.open(path) as image:
        image = image.convert('L').resize((200, 200))
        return np.asarray(image, dtype=np.float32).reshape(-1) / 255.0

def remove_training_outliers(split_df, threshold_sd=PCA_THRESHOLD_SD):
    keep = pd.Series(True, index=split_df.index)
    audit_rows = []
    train_df = split_df[split_df['split'] == 'train']

    for class_name, class_df in train_df.groupby('class_name'):
        X = np.stack([pca_vector(path) for path in class_df['path']])
        scores = PCA(n_components=2, svd_solver='randomized', random_state=42).fit_transform(X)
        means, stds = scores.mean(axis=0), scores.std(axis=0, ddof=0)
        outlier = np.any(np.abs(scores - means) > threshold_sd * np.maximum(stds, 1e-12), axis=1)
        keep.loc[class_df.index[outlier]] = False
        for idx, row_idx in enumerate(class_df.index):
            audit_rows.append({
                'row_index': row_idx, 'class_name': class_name,
                'pc1': scores[idx, 0], 'pc2': scores[idx, 1],
                'outlier': bool(outlier[idx])
            })

    cleaned = split_df.loc[keep].copy()
    audit = pd.DataFrame(audit_rows)
    return cleaned, audit


## 4. Datasets, models, and training

The ImageNet-pretrained convolutional base is frozen and only the classification head is trained. Training uses Adam, categorical cross-entropy, weight decay, early stopping, and a learning-rate scheduler. Augmentation is restricted to training images.


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class PathologyDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        with Image.open(row.path) as image:
            image = image.convert('RGB')
            tensor = self.transform(image)
        return tensor, int(row.label), row.patient_id, row.path

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def build_model(model_name, n_classes=5):
    if model_name == 'DenseNet-121':
        model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        for parameter in model.features.parameters(): parameter.requires_grad = False
        n_features = model.classifier.in_features
        model.classifier = nn.Sequential(nn.Dropout(0.30), nn.Linear(n_features, n_classes))
    elif model_name == 'AlexNet':
        model = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
        for parameter in model.features.parameters(): parameter.requires_grad = False
        for parameter in model.avgpool.parameters(): parameter.requires_grad = False
        n_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(n_features, n_classes)
    else:
        raise ValueError(model_name)
    return model.to(DEVICE)

def make_loaders(split_df):
    loaders = {}
    for split in ['train', 'validation', 'test']:
        transform = train_transform if split == 'train' else eval_transform
        ds = PathologyDataset(split_df[split_df.split == split], transform)
        loaders[split] = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=(split == 'train'),
                                    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    return loaders

@torch.no_grad()
def predict(model, loader):
    model.eval(); true, prob, patients, paths = [], [], [], []
    for x, y, patient, path in loader:
        p = torch.softmax(model(x.to(DEVICE)), dim=1).cpu().numpy()
        true.extend(y.numpy()); prob.extend(p); patients.extend(patient); paths.extend(path)
    return np.asarray(true), np.asarray(prob), np.asarray(patients), np.asarray(paths)

def train_one(model_name, split_df, seed):
    set_seed(seed)
    loaders = make_loaders(split_df)
    model = build_model(model_name)
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(trainable, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    criterion = nn.CrossEntropyLoss()
    history = []
    best_state, best_loss, stale = None, np.inf, 0

    for epoch in range(MAX_EPOCHS):
        model.train(); train_loss, train_correct, train_n = 0.0, 0, 0
        for x, y, _, _ in loaders['train']:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(); logits = model(x); loss = criterion(logits, y)
            loss.backward(); optimizer.step()
            train_loss += loss.item() * y.size(0)
            train_correct += (logits.argmax(1) == y).sum().item(); train_n += y.size(0)

        y_val, p_val, _, _ = predict(model, loaders['validation'])
        val_loss = log_loss(y_val, p_val, labels=np.arange(len(CLASS_NAMES)))
        row = {'epoch': epoch + 1, 'train_loss': train_loss/train_n,
               'train_accuracy': train_correct/train_n,
               'validation_loss': val_loss,
               'validation_accuracy': accuracy_score(y_val, p_val.argmax(1))}
        history.append(row); scheduler.step(val_loss)
        if val_loss < best_loss - 1e-4:
            best_loss, stale = val_loss, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            stale += 1
            if stale >= EARLY_STOPPING_PATIENCE: break

    model.load_state_dict(best_state)
    y_true, y_prob, patients, paths = predict(model, loaders['test'])
    return model, pd.DataFrame(history), y_true, y_prob, patients, paths


## 5. Medical-imaging metrics and patient-level confidence intervals

In [ ]:
def compute_metrics(y_true, y_prob):
    y_pred = y_prob.argmax(axis=1)
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(CLASS_NAMES)))
    precision, sensitivity, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=np.arange(len(CLASS_NAMES)), zero_division=0)
    rows = []
    for i, name in enumerate(CLASS_NAMES):
        tp = cm[i, i]; fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp; tn = cm.sum() - tp - fn - fp
        specificity = tn / (tn + fp) if (tn + fp) else np.nan
        rows.append({'class_name': name, 'precision': precision[i], 'sensitivity': sensitivity[i],
                     'specificity': specificity, 'f1_score': f1[i], 'support': support[i]})
    overall = {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'log_loss': log_loss(y_true, y_prob, labels=np.arange(len(CLASS_NAMES))),
        'roc_auc_ovr_macro': roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro'),
        'macro_precision': precision.mean(), 'macro_sensitivity': sensitivity.mean(),
        'macro_specificity': np.nanmean([r['specificity'] for r in rows]),
        'macro_f1': f1.mean(),
    }
    return overall, pd.DataFrame(rows), cm

def patient_bootstrap_ci(y_true, y_prob, patient_ids, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    unique_patients = np.unique(patient_ids)
    samples = []
    for _ in range(n_boot):
        sampled = rng.choice(unique_patients, size=len(unique_patients), replace=True)
        indices = np.concatenate([np.where(patient_ids == p)[0] for p in sampled])
        yt, yp = y_true[indices], y_prob[indices]
        if len(np.unique(yt)) < len(CLASS_NAMES): continue
        try:
            overall, _, _ = compute_metrics(yt, yp)
            samples.append(overall)
        except ValueError:
            continue
    boot = pd.DataFrame(samples)
    return pd.DataFrame({'lower_95': boot.quantile(.025), 'upper_95': boot.quantile(.975)})

def save_plots(history, y_true, y_prob, cm, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].plot(history.epoch, history.train_accuracy, label='Training')
    ax[0].plot(history.epoch, history.validation_accuracy, label='Validation')
    ax[0].set(xlabel='Epoch', ylabel='Accuracy', title='Accuracy'); ax[0].legend()
    ax[1].plot(history.epoch, history.train_loss, label='Training')
    ax[1].plot(history.epoch, history.validation_loss, label='Validation')
    ax[1].set(xlabel='Epoch', ylabel='Categorical cross-entropy', title='Log loss'); ax[1].legend()
    fig.tight_layout(); fig.savefig(output_dir / 'training_validation_curves.png', dpi=300); plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greys', xticklabels=CLASS_NAMES,
                yticklabels=CLASS_NAMES, ax=ax)
    ax.set(xlabel='Predicted label', ylabel='True label', title='Test confusion matrix')
    fig.tight_layout(); fig.savefig(output_dir / 'confusion_matrix.png', dpi=300); plt.close(fig)

    binary = label_binarize(y_true, classes=np.arange(len(CLASS_NAMES)))
    fig, ax = plt.subplots(figsize=(7, 6))
    for i, name in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(binary[:, i], y_prob[:, i])
        ax.plot(fpr, tpr, label=f'{name} (AUC={auc(fpr,tpr):.3f})')
    ax.plot([0,1], [0,1], '--', color='grey'); ax.set(xlabel='1 - Specificity', ylabel='Sensitivity')
    ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(output_dir / 'roc_curves.png', dpi=300); plt.close(fig)


## 6. Run a time-efficient set of repetitions

The principal experiment uses three repetitions for DenseNet-121 and AlexNet with PCA-based cleaning. The ablation adds three DenseNet-121 runs without PCA, for a total of nine training runs instead of 60. Each run is saved immediately so execution can be resumed safely.


In [ ]:
all_overall = []

for seed in range(N_REPEATS):
    base_split = split_patients(manifest, seed=seed)
    base_split.to_csv(OUTPUT_ROOT / f'patient_split_seed_{seed:02d}.csv', index=False)

    cleaned_split, pca_audit = remove_training_outliers(base_split)
    pca_audit.to_csv(OUTPUT_ROOT / f'pca_audit_seed_{seed:02d}.csv', index=False)

    experiment_specs = [
        (True, cleaned_split, ['DenseNet-121', 'AlexNet']),
        (False, base_split, ['DenseNet-121']),
    ]
    for use_pca, experiment_df, model_names in experiment_specs:
        condition = 'with_PCA' if use_pca else 'without_PCA'
        for model_name in model_names:
            run_dir = OUTPUT_ROOT / condition / model_name / f'seed_{seed:02d}'
            run_dir.mkdir(parents=True, exist_ok=True)
            completed = run_dir / 'overall_metrics.json'
            if completed.exists():
                with open(completed) as f: row = json.load(f)
                all_overall.append(row); continue

            model, history, y_true, y_prob, patients, paths = train_one(model_name, experiment_df, seed)
            overall, class_metrics, cm = compute_metrics(y_true, y_prob)
            ci = patient_bootstrap_ci(y_true, y_prob, patients, seed=seed)
            overall.update({'seed': seed, 'model': model_name, 'condition': condition,
                            'n_test_images': int(len(y_true)),
                            'n_test_patients': int(len(np.unique(patients)))})

            history.to_csv(run_dir / 'history.csv', index=False)
            class_metrics.to_csv(run_dir / 'class_metrics.csv', index=False)
            ci.to_csv(run_dir / 'patient_bootstrap_95CI.csv')
            pd.DataFrame({'path': paths, 'patient_id': patients, 'true_label': y_true,
                          'predicted_label': y_prob.argmax(1),
                          **{f'probability_{i}': y_prob[:, i] for i in range(len(CLASS_NAMES))}}
                        ).to_csv(run_dir / 'test_predictions.csv', index=False)
            save_plots(history, y_true, y_prob, cm, run_dir)
            with open(completed, 'w') as f: json.dump(overall, f, indent=2)
            all_overall.append(overall)
            print(condition, model_name, seed, overall['accuracy'])

results = pd.DataFrame(all_overall)
results.to_csv(OUTPUT_ROOT / 'all_repetitions_overall_metrics.csv', index=False)
display(results.groupby(['condition', 'model']).agg(['mean', 'std'])[
    ['accuracy', 'balanced_accuracy', 'log_loss', 'roc_auc_ovr_macro', 'macro_f1']])


## 7. Publication-ready summaries and formal comparisons

In [ ]:
from scipy.stats import wilcoxon

results = pd.read_csv(OUTPUT_ROOT / 'all_repetitions_overall_metrics.csv')
summary_metrics = ['accuracy', 'balanced_accuracy', 'log_loss', 'roc_auc_ovr_macro',
                   'macro_precision', 'macro_sensitivity', 'macro_specificity', 'macro_f1']
summary = results.groupby(['condition', 'model'])[summary_metrics].agg(['mean', 'std'])
summary.to_csv(OUTPUT_ROOT / 'summary_mean_sd.csv')

comparisons = []
for model_name in ['DenseNet-121']:
    a = results[(results.model == model_name) & (results.condition == 'with_PCA')].sort_values('seed')
    b = results[(results.model == model_name) & (results.condition == 'without_PCA')].sort_values('seed')
    for metric in summary_metrics:
        stat, p = wilcoxon(a[metric], b[metric])
        comparisons.append({'model': model_name, 'metric': metric,
                            'with_PCA_mean': a[metric].mean(),
                            'without_PCA_mean': b[metric].mean(),
                            'wilcoxon_statistic': stat, 'p_value': p})
comparisons = pd.DataFrame(comparisons)
comparisons.to_csv(OUTPUT_ROOT / 'PCA_ablation_wilcoxon.csv', index=False)

# Select the median-accuracy run for detailed figures; this avoids choosing the best run.
representative = []
for (condition, model_name), group in results.groupby(['condition', 'model']):
    median_accuracy = group.accuracy.median()
    row = group.iloc[(group.accuracy - median_accuracy).abs().argsort()[:1]].copy()
    representative.append(row)
representative = pd.concat(representative)
representative.to_csv(OUTPUT_ROOT / 'representative_median_runs.csv', index=False)

display(summary)
display(comparisons)
display(representative[['condition', 'model', 'seed', 'accuracy']])


## Outputs produced

- `manifest_40X.csv`: complete image/patient manifest;
- `patient_split_seed_XX.csv`: auditable patient-level assignment;
- `pca_audit_seed_XX.csv`: PCA scores and training outliers;
- training/validation accuracy and log-loss curves;
- test confusion matrices and ROC curves;
- class-wise sensitivity, specificity, precision, and F1-score;
- accuracy, balanced accuracy, log loss, macro ROC-AUC, and macro metrics;
- patient-level bootstrap 95% confidence intervals;
- mean ± SD across three repetitions;
- paired DenseNet-121 Wilcoxon ablation analysis with versus without PCA;
- median-run selection for publication figures.
